## introduction

```
You can use generation index directly (it’s proportional to evaluations).

Your JSON already has best_fitness as a best-so-far per generation list → use it.

Primary metric: final best-so-far at the common generation budget.

Tie-breaker: AUC (area under the best-so-far curve) up to that generation budget.

Aggregate across seeds per task, normalize per task (min–max), then average across tasks; break ties with CVaR-25%.

Below is a compact script for your file layout:

<ROOT>/<task>/<algo>/<config_id>/seed_<k>.json

```

In [1]:
import sys
sys.path.append('/home/ronedr/evolution-strategy-baselines-comparison')

In [20]:
import argparse
import json
import os
from dataclasses import dataclass
from glob import glob
from typing import List, Optional
import numpy as np
import pandas as pd

# Tasks where higher is better (e.g., reward/accuracy) – they’ll be negated
# You can list at either level: ("global_task", "specific_task") or just "global_task"
HIGHER_IS_BETTER = {
    ("GymnaxProblem", "Acrobot-v1"),
    ("GymnaxProblem", "CartPole-v1"),
}

# Optional time-to-target thresholds AFTER conversion to lower-is-better
TARGETS = {
    # ("bbob", "rastrigin"): 1e-6,
}

@dataclass
class RunKey:
    global_task: str
    specific_task: str
    algo: str
    config_id: str
    seed: str

def list_run_files(root: str) -> List[str]:
    # <root>/<global>/<specific>/<algo>/<config>/*.json
    return sorted(glob(os.path.join(root, "*", "*", "*", "*", "*.json")))

def parse_key(path: str) -> RunKey:
    parts = path.strip(os.sep).split(os.sep)
    global_task = parts[-5]
    specific_task = parts[-4]
    algo = parts[-3]
    config_id = parts[-2]
    seed = os.path.basename(path).split("/")[-1].split(".json")[0]
    return RunKey(global_task, specific_task, algo, config_id, seed)

def load_json(path: str) -> dict:
    with open(path, "r") as f:
        return json.load(f)

def is_higher_better(global_task: str, specific_task: str) -> bool:
    return (global_task, specific_task) in HIGHER_IS_BETTER or global_task in HIGHER_IS_BETTER

def to_lower_is_better(values: np.ndarray, higher_better: bool) -> np.ndarray:
    return -values if higher_better else values

def auc_under_curve(curve: np.ndarray) -> float:
    return float(np.trapz(curve, dx=1.0))  # over generations

def time_to_target(curve: np.ndarray, target: float) -> Optional[int]:
    idx = np.where(curve <= target)[0]
    return int(idx[0] + 1) if len(idx) else None

def cvar(values: List[float], alpha: float = 0.25) -> float:
    worst = sorted(values, reverse=True)
    k = max(1, int(np.ceil(alpha * len(worst))))
    return float(np.mean(worst[:k]))

def run(results_root, gen_budget, prefix):
    files = list_run_files(results_root)
    if not files:
        raise SystemExit("No runs found. Expected <root>/<global>/<specific>/<algo>/<config>/*.json")

    rows = []
    for fp in files:
        key = parse_key(fp)
        rec = load_json(fp)
        if "best_fitness" not in rec:
            continue

        curve = np.asarray(rec["best_fitness"], dtype=float)
        # enforce common generation budget
        if len(curve) >= gen_budget:
            curve = curve[:gen_budget]
        else:
            curve = np.pad(curve, (0, gen_budget - len(curve)), mode="edge")

        curve = to_lower_is_better(curve, is_higher_better(key.global_task, key.specific_task))
        curve = np.minimum.accumulate(curve)  # ensure best-so-far

        final_val = float(curve[-1])
        auc_val = auc_under_curve(curve)

        # optional per-task target
        tgt_key = (key.global_task, key.specific_task)
        target = TARGETS.get(tgt_key, TARGETS.get(key.global_task, None))
        ttt_val = time_to_target(curve, target) if target is not None else None

        rows.append({
            "global_task": key.global_task,
            "specific_task": key.specific_task,
            "algo": key.algo,
            "config_id": key.config_id,
            "seed": key.seed,
            "final_at_gen": final_val,
            "auc_best_curve": auc_val,
            "ttt": ttt_val
        })

    runs_df = pd.DataFrame(rows)
    if runs_df.empty:
        raise SystemExit("Parsed zero valid runs after reading JSONs.")

    # Aggregate across seeds per (global, specific, algo, config)
    per_specific = runs_df.groupby(
        ["global_task", "specific_task", "algo", "config_id"], as_index=False
    ).agg(
        final_mean=("final_at_gen", "mean"),
        final_std=("final_at_gen", "std"),
        final_median=("final_at_gen", "median"),
        auc_mean=("auc_best_curve", "mean"),
    )

    # Normalize PER specific task (min–max) so tasks are comparable
    eps = 1e-12
    per_specific["norm_final"] = per_specific.groupby(
        ["global_task", "specific_task"]
    )["final_mean"].transform(lambda s: (s - s.min()) / (s.max() - s.min() + eps))

    per_specific["norm_auc"] = per_specific.groupby(
        ["global_task", "specific_task"]
    )["auc_mean"].transform(lambda s: (s - s.min()) / (s.max() - s.min() + eps))

    # Global ranking across ALL specific tasks
    rows = []
    for (algo, config_id), sub in per_specific.groupby(["algo", "config_id"]):
        s_final = sub["norm_final"].tolist()
        s_auc = sub["norm_auc"].tolist()
        rows.append({
            "algo": algo,
            "config_id": config_id,
            "mean_norm_final": float(np.mean(s_final)),
            "cvar25_norm_final": cvar(s_final, 0.25),
            "mean_norm_auc": float(np.mean(s_auc)),
            "n_specific_tasks": int(sub[["global_task", "specific_task"]].drop_duplicates().shape[0]),
        })
    global_df = pd.DataFrame(rows).sort_values(
        by=["mean_norm_final", "cvar25_norm_final", "mean_norm_auc"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    # Per-family ranking (aggregate within each global_task)
    per_family = []
    for gt, sub in per_specific.groupby("global_task"):
        fam_rows = []
        for (algo, config_id), sub2 in sub.groupby(["algo", "config_id"]):
            s_final = sub2["norm_final"].tolist()
            s_auc = sub2["norm_auc"].tolist()
            fam_rows.append({
                "global_task": gt,
                "algo": algo,
                "config_id": config_id,
                "mean_norm_final": float(np.mean(s_final)),
                "cvar25_norm_final": cvar(s_final, 0.25),
                "mean_norm_auc": float(np.mean(s_auc)),
                "n_specific_tasks": int(sub2["specific_task"].nunique()),
            })
        fam_df = pd.DataFrame(fam_rows).sort_values(
            by=["mean_norm_final", "cvar25_norm_final", "mean_norm_auc"],
            ascending=[True, True, True]
        ).reset_index(drop=True)
        per_family.append(fam_df)
    per_family_df = pd.concat(per_family, ignore_index=True) if per_family else pd.DataFrame()

    # Save all
    os.makedirs("../reports", exist_ok=True)
    runs_df.to_csv(f"../reports/{prefix}_runs.csv", index=False)
    per_specific.to_csv(f"../reports/{prefix}_per_specific_task.csv", index=False)
    per_family_df.to_csv(f"../reports/{prefix}_per_family.csv", index=False)
    global_df.to_csv(f"../reports/{prefix}_global.csv", index=False)

    print("=== Global best configs (lower is better) ===")
    print(global_df.head(10).to_string(index=False))
    print("\n=== Per-family best (lower is better) ===")
    if not per_family_df.empty:
        for gt in per_family_df["global_task"].unique():
            top = per_family_df[per_family_df["global_task"] == gt].head(5)
            print(f"\n[{gt}]")
            print(top.to_string(index=False))
    print("\nSaved CSVs in ../reports/")

In [21]:
results_root = "../tuning_results"
gen_budget = 1000
prefix = "hier"
run(results_root, gen_budget, prefix)

=== Global best configs (lower is better) ===
    algo config_id  mean_norm_final  cvar25_norm_final  mean_norm_auc  n_specific_tasks
SimpleES  config_1            0.000               0.00       0.478859                 2
SimpleES  config_2            0.000               0.00       0.901163                 2
SimpleES  config_3            0.125               0.25       0.338453                 2
  CMA_ES  config_3            0.250               0.50       0.356880                 2
  CMA_ES  config_2            0.500               1.00       0.287110                 2
  CMA_ES  config_1            0.500               1.00       0.677956                 2

=== Per-family best (lower is better) ===

[GymnaxProblem]
  global_task     algo config_id  mean_norm_final  cvar25_norm_final  mean_norm_auc  n_specific_tasks
GymnaxProblem SimpleES  config_1            0.000               0.00       0.478859                 2
GymnaxProblem SimpleES  config_2            0.000               0.00      